# 01 — Data Understanding
فحص ملفات CSV، الأبعاد، الأنواع، القيم المفقودة، التكرارات، والمفاتيح المحتملة قبل أي تنظيف.

In [1]:
from pathlib import Path
import pandas as pd

RAW = Path('../data/raw')
expected = ['customers.csv', 'accounts.csv', 'transactions.csv', 'branches.csv', 'products.csv']
available = [p.name for p in RAW.glob('*.csv')]
missing = sorted(set(expected) - set(available))
print(f'CSV files found: {len(available)}')
print('Missing expected files:', missing or 'None')

CSV files found: 9
Missing expected files: None


In [2]:
tables = {}
for path in sorted(RAW.glob('*.csv')):
    try:
        tables[path.stem] = pd.read_csv(path)
    except Exception as exc:
        print(f'Could not read {path.name}: {exc}')

overview = []
for name, frame in tables.items():
    overview.append({
        'table': name,
        'rows': len(frame),
        'columns': len(frame.columns),
        'duplicate_rows': int(frame.duplicated().sum()),
        'missing_cells': int(frame.isna().sum().sum()),
    })
pd.DataFrame(overview)

,table,rows,columns,duplicate_rows,missing_cells
0,accounts,600,14,0,322
1,branches,12,4,0,0
2,cards,358,13,0,0
3,customers,200,21,0,0
4,loans,95,13,0,0
5,merchants,254,9,0,0
6,products,24,4,0,0
7,subscriptions,1333,12,0,1134
8,transactions,476202,30,0,687045


In [3]:
for name, frame in tables.items():
    print(f'\n--- {name} ---')
    display(pd.DataFrame({'dtype': frame.dtypes.astype(str), 'missing': frame.isna().sum(), 'unique': frame.nunique(dropna=True)}))
    display(frame.head(3))


--- accounts ---


,dtype,missing,unique
account_id,object,0,600
customer_id,object,0,200
iban,object,0,600
bic,object,322,8
bank_name,object,0,12
account_type,object,0,24
role,object,0,7
opened_at,object,0,530
initial_balance,float64,0,600
current_balance,float64,0,600


,account_id,customer_id,iban,bic,bank_name,account_type,role,opened_at,initial_balance,current_balance,overdraft_limit,status,branch_id,product_id
0,A00000001,C0000001,FR763000271023RU98WRUTXJ652,CRSYFRPPXXX,Crédit Synthétique,compte_courant,CURRENT,2005-11-01,7495.69,-3248.088344,2500.0,active,5,7
1,A00000002,C0000001,FR7630002624617YT1GY9I2Y486,CRSYFRPPXXX,Crédit Synthétique,livret_a,SAVINGS,2006-11-24,5497.94,38019.530000,0.0,active,5,18
2,A00000003,C0000001,FR763000222777R8EKCSH7UL282,CRSYFRPPXXX,Crédit Synthétique,assurance_vie,RETIREMENT,2011-07-01,291436.09,291436.090000,0.0,active,5,1



--- branches ---


,dtype,missing,unique
branch_id,int64,0,12
branch_name,object,0,12
city,object,0,1
source_note,object,0,1


...

,dtype,missing,unique
subscription_id,object,0,1333
customer_id,object,0,200
account_id,object,0,214
merchant_name,object,0,65
category,object,0,8
subcategory,object,0,27
amount,float64,0,939
frequency_months,int64,0,2
billing_day,int64,0,28
start_date,object,0,876


,subscription_id,customer_id,account_id,merchant_name,category,subcategory,amount,frequency_months,billing_day,start_date,end_date,status
0,S00000001,C0000001,A00000001,Amazon Prime,abonnements,streaming_video,6.99,1,8,2024-05-31,NaN,active
1,S00000002,C0000001,A00000001,Engie,energie,gaz,30.00,1,7,2024-07-19,2026-03-04,churned
2,S00000003,C0000001,A00000001,Orange,telecom,mobile,22.91,1,2,2023-09-13,NaN,active



--- transactions ---


,dtype,missing,unique
transaction_id,object,0,476202
account_id,object,0,352
customer_id,object,0,200
transaction_date,object,0,469921
amount,float64,0,40684
currency,object,0,3
merchant_name,object,0,473
merchant_category,object,0,24
merchant_mcc,int64,0,44
transaction_type,object,0,3


,transaction_id,account_id,customer_id,transaction_date,amount,currency,merchant_name,merchant_category,merchant_mcc,transaction_type,...,is_cash_withdrawal,balance_after_transaction,is_foreign,original_amount,original_currency,fx_rate,foreign_fee,signed_amount,source_transaction_type,transaction_status
0,70cb19af0c5347bb9d69a2cd6f1918a7,A00000001,C0000001,2023-10-07T12:36:07,4.19,EUR,Brioche Dorée,restauration,5814,Withdrawal,...,False,7491.50,False,-4.19,EUR,1.0,0.0,-4.19,card_payment_contactless,Completed
1,839148d0a1624fb88a88ef9ea5569175,A00000001,C0000001,2023-10-07T09:06:50,34.24,EUR,Lidl,alimentation,5411,Withdrawal,...,False,7457.26,False,-34.24,EUR,1.0,0.0,-34.24,card_payment_contactless,Completed
2,a2d4a6fbc01f418e8a87a8fed7d3b1e4,A00000001,C0000001,2023-10-07T16:17:36,69.67,EUR,Intermarché,alimentation,5411,Withdrawal,...,False,7387.59,False,-69.67,EUR,1.0,0.0,-69.67,card_payment_contactless,Completed


## Review checklist
- Confirm primary keys are unique and non-null.
- Confirm foreign keys match parent tables.
- Identify date, amount, status, and category columns from the actual schema.
- Record any schema differences before editing notebook 02.